# FP_NB_Gold_Model
**Northbrook Lending – Silver ➜ Gold star schema**

Builds the reporting layer used by the `FP_Northbrook_Model` semantic model (Direct Lake).

| Table | Grain | Notes |
|---|---|---|
| `DimDate` | one row per calendar day (2015–2027) | UK financial year (Apr–Mar), month-end flags |
| `DimCustomer` | one row per customer **version** (SCD2) | `CustomerSK = CustomerID × 100 + version`; bands for age, income and credit score. **Direct identifiers (name parts, e-mail, phone, street address, date of birth) are excluded on purpose**: data minimisation |
| `DimProduct`, `DimBranch` | one row per product / branch | include an `Unknown (-1)` member |
| `FactRepayment` | one row per instalment due | point-in-time customer key (the customer version valid on the due date) |
| `FactLoanSnapshot` | one row per loan per month-end (last 12 month-ends + as-at date) | outstanding balance, days past due, arrears bucket |
| `AggArrearsCollectionsMonthly` | month-end × product type | arrears and collections summary for fast trend visuals |

Balances are recalculated from the repayment history with the standard annuity formula, so each snapshot reflects what was actually paid by that date.


In [1]:
# Parameters - the pipeline overrides these at run time; run_id stays empty when you run the notebook by hand
run_id = ""
as_at_date = "2026-11-19"    # reporting "as at" date (matches the synthetic data generator)

StatementMeta(, 6518c219-7a95-45b9-91ec-08dc8a38e5fa, 3, Finished, Available, Finished, False)

In [2]:
# ---------- Setup ----------
# uuid for run IDs, json for the exit value, datetime for date handling
import uuid, json, datetime as dt
# Spark SQL functions such as col, when and date_format - imported as F
from pyspark.sql import functions as F

# Use the pipeline run ID if one was passed in, otherwise make a random one for manual runs
run_id = run_id or str(uuid.uuid4())
# Turn the as-at date text into a real date
AS_AT = dt.date.fromisoformat(as_at_date)

# Find the OneLake root folder of a lakehouse by name, in whichever workspace this runs
def lakehouse_root(name: str) -> str:
    try:
        # Preferred - ask Fabric for this lakehouse by name
        lh = notebookutils.lakehouse.get(name)
        # The lookup can return a dict or an object - read its properties either way
        props = lh["properties"] if isinstance(lh, dict) else lh.properties
        # Take its OneLake Tables path and cut off /Tables to leave the lakehouse root
        return props["oneLakeTablesPath"].rstrip("/").rsplit("/Tables", 1)[0]
    except Exception:
        # Fallback if the lookup fails - build the OneLake path from the current workspace name
        ws = notebookutils.runtime.context["currentWorkspaceName"]
        return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{name}.Lakehouse"

# Build the full path to a Delta table - root/Tables/dbo/TableName
def table_path(root, table, schema="dbo"):
    return f"{root}/Tables/{schema}/{table}"

# Find the Silver and Gold lakehouse roots once for the whole run
SILVER, GOLD = lakehouse_root("FP_Silver"), lakehouse_root("FP_Gold")
# Shortcut - read a Silver table by name
silver = lambda t: spark.read.format("delta").load(table_path(SILVER, t))
# Row count of each Gold table written, collected for the run summary
written = {}

# Save a DataFrame as a Gold Delta table and report its row count
def save(df, name):
    # Overwrite the Gold table; overwriteSchema allows its columns to change
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(table_path(GOLD, name))
    # Re-read the saved table to count its rows, record the count, then print it
    written[name] = spark.read.format("delta").load(table_path(GOLD, name)).count()
    print(f"{name:<30} {written[name]:>9,} rows")

# Add an Unknown member with key -1, so facts whose lookup fails still join to a dimension row
def with_unknown(dim, key, label_cols):
    """Append an 'Unknown' member (key -1) so facts never lose rows on a failed lookup."""
    # Make one row with the same columns as the dimension, every value NULL
    u = dim.limit(1).select([F.lit(None).cast(f.dataType).alias(f.name) for f in dim.schema.fields])
    # Give it key -1, keeping the same data type as the real keys
    u = u.withColumn(key, F.lit(-1).cast(dim.schema[key].dataType))
    # Show Unknown in each label column
    for c in label_cols:
        u = u.withColumn(c, F.lit("Unknown"))
    # Add the Unknown row to the bottom of the dimension
    return dim.unionByName(u)

# Turn a date into a whole-number key like 20261119, used to join facts to DimDate
date_key = lambda c: F.date_format(c, "yyyyMMdd").cast("int")

StatementMeta(, 6518c219-7a95-45b9-91ec-08dc8a38e5fa, 4, Finished, Available, Finished, False)

In [3]:
# ---------- Dimensions ----------
# DimDate
# Financial year starts in April - January to March belong to the previous year
fy_start = F.when(F.month("Date") >= 4, F.year("Date")).otherwise(F.year("Date") - 1)
# One row per calendar day from 2015 to 2027, then add the calendar columns
dim_date = (spark.sql("SELECT explode(sequence(to_date('2015-01-01'), to_date('2027-12-31'), interval 1 day)) AS Date")
    # Whole-number key like 20261119, used to join to the facts
    .withColumn("DateKey", date_key("Date"))
    .withColumn("Year", F.year("Date"))
    # Calendar labels - quarter, month number and names, and year-month
    .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Date").cast("string")))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("MonthShort", F.date_format("Date", "MMM"))
    .withColumn("YearMonth", F.date_format("Date", "yyyy-MM"))
    # First and last day of the month
    .withColumn("MonthStart", F.trunc("Date", "month"))
    .withColumn("MonthEnd", F.last_day("Date"))
    # True on the last day of each month - used for month-end snapshots
    .withColumn("IsMonthEnd", F.col("Date") == F.last_day("Date"))
    .withColumn("DayOfWeek", (F.dayofweek("Date") + 5) % 7 + 1)          # 1 = Monday
    .withColumn("DayName", F.date_format("Date", "EEEE"))
    # Weekend flag - Spark counts Sunday as 1 and Saturday as 7
    .withColumn("IsWeekend", F.dayofweek("Date").isin(1, 7))
    # UK financial year label, e.g. FY2026/27
    .withColumn("FinancialYear", F.format_string("FY%d/%02d", fy_start, (fy_start + 1) % 100)))
# Save the date dimension to Gold
save(dim_date, "DimDate")

# DimProduct / DimBranch (natural integer keys are stable in the source, so they double as keys)
# Product dimension - ProductID becomes ProductKey
dim_product = silver("Product").select(
    F.col("ProductID").alias("ProductKey"), "ProductCode", "ProductName", "ProductType",
    # Flag secured lending - Secured and Mortgage products
    F.col("ProductType").isin("Secured", "Mortgage").alias("IsSecured"),
    "BaseRatePct", "MinAmount", "MaxAmount", "MinTermMonths", "MaxTermMonths")
# Add the Unknown member and save as DimProduct
save(with_unknown(dim_product, "ProductKey", ["ProductCode", "ProductName", "ProductType"]), "DimProduct")

# Branch dimension - BranchID becomes BranchKey
dim_branch = silver("Branch").select(
    F.col("BranchID").alias("BranchKey"), "BranchCode", "BranchName", "Town", "Region", "Postcode", "OpenedDate")
# Add the Unknown member and save as DimBranch
save(with_unknown(dim_branch, "BranchKey", ["BranchCode", "BranchName", "Town", "Region"]), "DimBranch")

# DimCustomer (SCD2 history; no direct identifiers)
# Age in whole years on the as-at date
age = F.floor(F.months_between(F.lit(AS_AT), F.col("DateOfBirth")) / 12).cast("int")
# Build the customer dimension from Silver, one row per customer version
dim_customer = silver("Customer").select(
    # Surrogate key - CustomerID times 100 plus version, so every customer version is unique
    (F.col("CustomerID").cast("long") * 100 + F.col("CustomerVersion")).alias("CustomerSK"),
    # Location kept at town level; names, e-mail, phone, street address and date of birth are left out on purpose
    "CustomerID", "CustomerRef", "Town", "Region",
    # Postcode area - the leading letters, e.g. WS
    F.regexp_extract("Postcode", r"^([A-Z]{1,2})", 1).alias("PostcodeArea"),
    # Postcode district - the part before the space, e.g. WS1
    F.split("Postcode", " ").getItem(0).alias("PostcodeDistrict"),
    "EmploymentStatus", "AnnualIncome",
    # Income bands - numbered so they sort in order
    F.when(F.col("AnnualIncome") < 20000, "1. Under £20k").when(F.col("AnnualIncome") < 35000, "2. £20k–£35k")
     .when(F.col("AnnualIncome") < 50000, "3. £35k–£50k").when(F.col("AnnualIncome") < 75000, "4. £50k–£75k")
     .otherwise("5. £75k+").alias("IncomeBand"),
    "CreditScore",
    # Credit score bands - numbered so they sort in order
    F.when(F.col("CreditScore") < 500, "1. Very poor").when(F.col("CreditScore") < 600, "2. Poor")
     .when(F.col("CreditScore") < 700, "3. Fair").when(F.col("CreditScore") < 800, "4. Good")
     .otherwise("5. Excellent").alias("CreditBand"),
    # Age on the as-at date
    age.alias("AgeAtAsAt"),
    # Age bands
    F.when(age < 25, "18–24").when(age < 35, "25–34").when(age < 45, "35–44")
     .when(age < 55, "45–54").when(age < 65, "55–64").otherwise("65+").alias("AgeBand"),
    # Keep the SCD2 history columns, so each customer version has its own row and date range
    "CustomerSince", "CustomerVersion", "ValidFrom", "ValidTo", "IsCurrent")
# Add the Unknown member, key -1
dim_customer = with_unknown(dim_customer, "CustomerSK", ["CustomerRef", "Town", "Region", "EmploymentStatus"])
# Save the customer dimension to Gold
save(dim_customer, "DimCustomer")

StatementMeta(, 6518c219-7a95-45b9-91ec-08dc8a38e5fa, 5, Finished, Available, Finished, False)

DimDate                            4,748 rows
DimProduct                             9 rows
DimBranch                             13 rows
DimCustomer                        5,001 rows


In [4]:
# ---------- Facts ----------
# Read the Silver tables the facts are built from
loans = silver("Loan")
reps = silver("Repayment")
arrears = silver("Arrears")
# Product type for each product, used in the monthly summary
products = silver("Product").select("ProductID", "ProductType")
# Every real customer version from DimCustomer, leaving out the Unknown row, with its date range
cust_hist = (spark.read.format("delta").load(table_path(GOLD, "DimCustomer")).filter("CustomerSK <> -1")
                  .select(F.col("CustomerID").alias("_cid"), F.col("CustomerSK").alias("_pit_sk"),
                          "ValidFrom", "ValidTo", "IsCurrent"))
# Current version of each customer - the fallback when no version matches the date
cust_current = cust_hist.filter("IsCurrent").select("_cid", F.col("_pit_sk").alias("_cur_sk"))

# Add the right CustomerSK to each fact row - the customer version that was valid on a given date
def customer_sk_at(df, date_col):
    """Point-in-time lookup: the customer version valid on date_col (falls back to current, then -1)."""
    # Treat the date as a timestamp so it compares cleanly with ValidFrom and ValidTo
    ts = F.col(date_col).cast("timestamp")
    # Customer versions with their valid-from and valid-to dates
    pit = cust_hist.select("_cid", "_pit_sk", "ValidFrom", "ValidTo")
    # Join to the customer version whose date range covers the date
    return (df.join(pit, (F.col("CustomerID") == F.col("_cid")) & (ts >= F.col("ValidFrom")) & (ts < F.col("ValidTo")), "left")
              # Drop the helper columns from that join
              .drop("_cid", "ValidFrom", "ValidTo")
              # Also look up the current version, as a fallback when no version covers the date
              .join(cust_current, F.col("CustomerID") == F.col("_cid"), "left").drop("_cid")
              # Use the matched version, else the current one, else -1 for Unknown
              .withColumn("CustomerSK", F.coalesce("_pit_sk", "_cur_sk", F.lit(-1).cast("long")))
              .drop("_pit_sk", "_cur_sk"))

# FactRepayment – one row per instalment due
# Join each repayment to its loan to pick up the customer, product and branch
fr = (reps.join(loans.select("LoanID", "LoanRef", "CustomerID", "ProductID", "BranchID"), "LoanID")
    # Days late - paid date minus due date, or days overdue at the as-at date if unpaid; never below zero
    .withColumn("DaysLate", F.when(F.col("PaidDate").isNotNull(), F.greatest(F.datediff("PaidDate", "DueDate"), F.lit(0)))
                             .otherwise(F.greatest(F.datediff(F.lit(AS_AT), "DueDate"), F.lit(0))))
    # Amount still owed on the instalment
    .withColumn("ShortfallAmount", (F.col("DueAmount") - F.col("PaidAmount")).cast("decimal(18,2)"))
    # 1 or 0 status flags, so they can be summed in reports
    .withColumn("IsOnTime", (F.col("Status") == "Paid").cast("int"))
    .withColumn("IsLate", (F.col("Status") == "Late").cast("int"))
    .withColumn("IsMissed", F.col("Status").isin("Missed", "Partial").cast("int"))
    # Date keys for the due and paid dates; -1 when not yet paid
    .withColumn("DueDateKey", date_key("DueDate"))
    .withColumn("PaidDateKey", F.coalesce(date_key("PaidDate"), F.lit(-1))))
# Attach the customer version that was valid on each due date
fact_repayment = customer_sk_at(fr, "DueDate").select(
    # Keep the fact columns; IDs are renamed to keys that join to the dimensions
    "RepaymentID", "LoanID", "LoanRef", "CustomerSK",
    F.col("ProductID").alias("ProductKey"), F.col("BranchID").alias("BranchKey"),
    "DueDateKey", "PaidDateKey", "InstalmentNumber", F.col("Status").alias("RepaymentStatus"), "PaymentMethod",
    "DueAmount", "PaidAmount", "ShortfallAmount", "DaysLate", "IsOnTime", "IsLate", "IsMissed")
# Save the repayment fact to Gold
save(fact_repayment, "FactRepayment")

# FactLoanSnapshot – loan x month-end (last 12 month-ends + as-at date)
# Start from the last day of the month before the as-at date
snaps, d = [], AS_AT.replace(day=1) - dt.timedelta(days=1)
# Collect 12 month-ends, stepping back one month each time
for _ in range(12):
    snaps.append(d)
    d = d.replace(day=1) - dt.timedelta(days=1)
# Add the as-at date itself as the latest snapshot
snaps.append(AS_AT)
# Turn the list of dates into a one-column Spark table
snap_df = spark.createDataFrame([(s,) for s in snaps], "SnapshotDate date")

# One row per loan per snapshot date, from the date the loan started
ls = loans.crossJoin(snap_df).filter(F.col("StartDate") <= F.col("SnapshotDate"))
# Count instalments paid, on time or late, by each snapshot date
paid = (reps.filter(F.col("Status").isin("Paid", "Late")).join(snap_df, F.col("PaidDate") <= F.col("SnapshotDate"))
            .groupBy("LoanID", "SnapshotDate").agg(F.count("*").alias("PaidInstalments")))
# Add the paid count, 0 if none, and the arrears position for each loan and snapshot date
ls = (ls.join(paid, ["LoanID", "SnapshotDate"], "left").fillna({"PaidInstalments": 0})
        .join(arrears.select("LoanID", "SnapshotDate", "DaysPastDue", "ArrearsAmount", "ArrearsBucket", "MissedInstalments"),
              ["LoanID", "SnapshotDate"], "left"))

# Monthly interest rate as a fraction - the annual percentage divided by 12 then by 100
r = F.col("InterestRatePct").cast("double") / 1200
# k = number of instalments paid by the snapshot date
k = F.col("PaidInstalments").cast("double")
# How much interest has compounded over those k months
growth = F.pow(1 + r, k)
# Standard annuity formula - the balance left after k monthly payments
balance = F.col("PrincipalAmount").cast("double") * growth - F.col("MonthlyPayment").cast("double") * (growth - 1) / r
# A loan is closed once every instalment has been paid
is_closed = F.col("PaidInstalments") >= F.col("TermMonths")

# Add the balance, arrears and status measures to each loan snapshot row
fls = (ls.withColumn("IsOpen", (~is_closed).cast("int"))
         # Balance still owed - zero once the loan is closed, and never below zero
         .withColumn("OutstandingBalance", F.when(is_closed, 0).otherwise(F.greatest(balance, F.lit(0))).cast("decimal(18,2)"))
         # Fill gaps in the arrears details - loans with no arrears record show zero and Up to date
         .withColumn("DaysPastDue", F.coalesce("DaysPastDue", F.lit(0)))
         .withColumn("ArrearsAmount", F.coalesce("ArrearsAmount", F.lit(0)).cast("decimal(18,2)"))
         .withColumn("MissedInstalments", F.coalesce("MissedInstalments", F.lit(0)))
         .withColumn("ArrearsBucket", F.coalesce("ArrearsBucket", F.lit("Up to date")))
         # 1 if any payment is overdue
         .withColumn("IsInArrears", (F.col("DaysPastDue") > 0).cast("int"))
         # Days-past-due flags at 30, 60 and 90 days
         .withColumn("Is30Plus", (F.col("DaysPastDue") >= 30).cast("int"))
         .withColumn("Is60Plus", (F.col("DaysPastDue") >= 60).cast("int"))
         .withColumn("Is90Plus", (F.col("DaysPastDue") >= 90).cast("int"))
         # Loan status on the snapshot date - Closed, Default at 90+ days past due, Arrears, or Up to date
         .withColumn("LoanStatusAtSnapshot", F.when(is_closed, "Closed").when(F.col("DaysPastDue") >= 90, "Default")
                                              .when(F.col("DaysPastDue") > 0, "Arrears").otherwise("Up to date"))
         # Date key for the snapshot date
         .withColumn("SnapshotDateKey", date_key("SnapshotDate")))
# Attach the customer version that was valid on each snapshot date
fact_loan_snapshot = customer_sk_at(fls, "SnapshotDate").select(
    # Keep the fact columns; IDs are renamed to keys that join to the dimensions
    "SnapshotDateKey", "SnapshotDate", "LoanID", "LoanRef", "CustomerSK",
    F.col("ProductID").alias("ProductKey"), F.col("BranchID").alias("BranchKey"),
    "PrincipalAmount", "InterestRatePct", "MonthlyPayment", "TermMonths", "PaidInstalments",
    "OutstandingBalance", "ArrearsAmount", "DaysPastDue", "MissedInstalments", "ArrearsBucket",
    "LoanStatusAtSnapshot", "IsOpen", "IsInArrears", "Is30Plus", "Is60Plus", "Is90Plus")
# Save the loan snapshot fact to Gold
save(fact_loan_snapshot, "FactLoanSnapshot")

StatementMeta(, 6518c219-7a95-45b9-91ec-08dc8a38e5fa, 6, Finished, Available, Finished, False)

FactRepayment                     59,236 rows
FactLoanSnapshot                  49,808 rows


In [5]:
# ---------- Monthly arrears & collections summary (month-end x product type) ----------
# Read the loan snapshot fact back from Gold and add each loan's product type
fls_g = spark.read.format("delta").load(table_path(GOLD, "FactLoanSnapshot")).join(
    products, F.col("ProductKey") == F.col("ProductID")).drop("ProductID")
# Portfolio position per snapshot date and product type
portfolio = fls_g.groupBy("SnapshotDate", "ProductType").agg(
    # Open loans, balance owed, loans and balance in arrears, and loans 30, 60 and 90+ days past due
    F.sum("IsOpen").alias("OpenLoans"),
    F.sum("OutstandingBalance").alias("OutstandingBalance"),
    F.sum("IsInArrears").alias("LoansInArrears"),
    F.sum("ArrearsAmount").alias("ArrearsBalance"),
    F.sum("Is30Plus").alias("Loans30Plus"),
    F.sum("Is60Plus").alias("Loans60Plus"),
    F.sum("Is90Plus").alias("Loans90Plus"))

# collections: instalments falling due in the month up to the snapshot date, and how much was collected by then
# Add the product type to each repayment through its loan
due = (reps.join(loans.select("LoanID", "ProductID"), "LoanID").join(products, "ProductID")
           # Keep instalments due between the start of the snapshot month and the snapshot date
           .join(snap_df, (F.col("DueDate") >= F.trunc("SnapshotDate", "month")) & (F.col("DueDate") <= F.col("SnapshotDate")))
           # Total due and total collected per snapshot date and product type
           .groupBy("SnapshotDate", "ProductType").agg(
               F.sum("DueAmount").alias("AmountDue"),
               # A payment only counts as collected if it was paid by the snapshot date
               F.sum(F.when(F.col("PaidDate") <= F.col("SnapshotDate"), F.col("PaidAmount")).otherwise(0)).alias("AmountCollected")))

# Combine the portfolio and collections figures; months with nothing due show zero
agg = (portfolio.join(due, ["SnapshotDate", "ProductType"], "left").fillna(0, ["AmountDue", "AmountCollected"])
    # Date key and year-month label for each snapshot
    .withColumn("SnapshotDateKey", date_key("SnapshotDate"))
    .withColumn("YearMonth", F.date_format("SnapshotDate", "yyyy-MM"))
    # Arrears rate = share of open loans in arrears; collections rate = share of the amount due that was collected; both as percentages
    .withColumn("ArrearsRatePct", F.round(F.when(F.col("OpenLoans") > 0, F.col("LoansInArrears") / F.col("OpenLoans") * 100), 2))
    .withColumn("CollectionsRatePct", F.round(F.when(F.col("AmountDue") > 0, F.col("AmountCollected") / F.col("AmountDue") * 100), 2)))
# Save the monthly summary to Gold
save(agg, "AggArrearsCollectionsMonthly")

# Quick check - show the whole-portfolio trend by month: open loans, balance in GBP millions, arrears and collections rates
display(spark.read.format("delta").load(table_path(GOLD, "AggArrearsCollectionsMonthly"))
             .groupBy("SnapshotDate").agg(F.sum("OpenLoans").alias("OpenLoans"),
                                          F.round(F.sum("OutstandingBalance") / 1e6, 1).alias("OutstandingGBPm"),
                                          F.sum("LoansInArrears").alias("LoansInArrears"),
                                          F.round(F.sum("LoansInArrears") / F.sum("OpenLoans") * 100, 2).alias("ArrearsRatePct"),
                                          F.round(F.sum("AmountCollected") / F.sum("AmountDue") * 100, 2).alias("CollectionsRatePct"))
             .orderBy("SnapshotDate"))

StatementMeta(, 6518c219-7a95-45b9-91ec-08dc8a38e5fa, 7, Finished, Available, Finished, False)

AggArrearsCollectionsMonthly          52 rows


SynapseWidget(Synapse.DataFrame, 525c9e3b-251e-45b4-b080-0132c2959a88)

In [6]:
# End the notebook and hand a JSON summary back to the calling pipeline - run ID, status and row count per Gold table
notebookutils.notebook.exit(json.dumps({"notebook": "FP_NB_Gold_Model", "run_id": run_id,
                                        "status": "Succeeded", "tables": written}))

StatementMeta(, 6518c219-7a95-45b9-91ec-08dc8a38e5fa, 8, Finished, Available, Finished, False)

ExitValue: {"notebook": "FP_NB_Gold_Model", "run_id": "2a7916bb-1075-487b-b629-ddf2bb31807b", "status": "Succeeded", "tables": {"DimDate": 4748, "DimProduct": 9, "DimBranch": 13, "DimCustomer": 5001, "FactRepayment": 59236, "FactLoanSnapshot": 49808, "AggArrearsCollectionsMonthly": 52}}